# Inverse Folding (Ubiquitin)

**BioPipelines example** — inverse folding of ubiquitin using ProteinMPNN. Designed sequences are refolded with AlphaFold2 and filtered by RMSD and pLDDT. Sequences passing the filter are codon-optimised for *E. coli* expression with DNAEncoder.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold

with Pipeline(project="Setup", job="InstallTools"):
    RFdiffusion.install(weights=[])  # SE3nv environment only, no weights (required by ProteinMPNN)
    ProteinMPNN.install()
    AlphaFold.install()

## Cell 3: Inverse Folding Pipeline

Fetches ubiquitin (PDB: 4LCD, chain E) and generates 10 sequences with ProteinMPNN (soluble model).
AlphaFold2 refolds each sequence; those with RMSD < 1.5 Å and pLDDT > 80 are codon-optimised for *E. coli* expression.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.dna_encoder import DNAEncoder

with Pipeline(project="Ubiquitin", job="InverseFolding"):
    ubiquitin = PDB("4LCD", chain="E")
    sequences = ProteinMPNN(structures=ubiquitin,
                            num_sequences=10,
                            soluble_model=True)
    folded = AlphaFold(proteins=sequences)
    dna = DNAEncoder(sequences=sequences,
                     organism="EC")
    conf_change = ConformationalChange(reference_structures=ubiquitin,
                                       target_structures=folded)
    filtered_sequences = Panda(tables=[folded.tables.confidence,
                                       conf_change.tables.changes],
                               operations=[Panda.merge(),
                                            Panda.filter("RMSD < 1.5 and plddt > 80")],
                               pool=sequences)
    dna = DNAEncoder(sequences=filtered_sequences,
                     organism="EC")
dna